In [2]:
import cv2
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
print(cv2.__version__)

4.12.0


In [3]:
eye_cascade = cv2.CascadeClassifier('haarcascade_eye.xml')

In [4]:
left = []
right = []
img_arr = [i for i in range(0,5)]


for i in range(1,6):
    img = cv2.imread('img/train'+str(i)+'.jpg')
    img_arr[i-1] = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    file = open('img/train' + str(i) + '.gnd', 'r')
    l = file.readline()
    r = file.readline()

    (glx, gly) = int(l.split(" ")[1].split(',')[0]), int(l.split(" ")[1].split(',')[1])
    (grx, gry) = int(r.split(" ")[1].split(',')[0]), int(r.split(" ")[1].split(',')[1])

    #print(glx, gly)
    #print(grx, gry)

    left.append((glx,gly))
    right.append((grx,gry))

In [5]:
def measureDistance(lx,ly, glx,gly,  rx,ry, grx,gry):
    dl, dr = 10.0,10.0

    # applied pythagorean theorem to calculate the distance
    
    if lx is not None and ly is not None:
        dl = ((glx - lx)**2 + (gly - ly)**2) ** .5

    if rx is not None and ry is not None:
        dr = ((grx - rx) ** 2 + (gry - ry) ** 2) ** .5

    return dl, dr

In [21]:
# create a function to detect eyes
#######################################
# Problem 6: default values for scaleFactor and minNeighbors are optimized values from Problem 5
#######################################
def detect_eyes(img, scaleFactor=1.84, minNeighbors=10):
    eye_img = img.copy()
    height, width, channels = eye_img.shape

    eye_rect = eye_cascade.detectMultiScale(eye_img, scaleFactor=scaleFactor, minNeighbors=minNeighbors,
                                            minSize=(int(width * 0.1), int(height * 0.1)))

    #print(eye_rect)
    eye_rect = sorted(eye_rect, key=lambda e: e[2]*e[3], reverse=False)
    #print('sorted eye_rect', eye_rect)
    eye_rect = eye_rect[:2]

    results = []
    for (x,y,w,h) in eye_rect:
        # red box for eyes
        cv2.rectangle(eye_img, (x,y), (x+w, y+h), (255,0,0), 1)
        cx, cy = (x+w)//2, (y+h)/2

        results.append(cx)
        results.append(cy)


    return results

In [ ]:
# 3d array to store results (avg distances)
r = [[[0 for i in range(10)] for j in range(100)] for k in range(5)]


for i in range(0,5):
    for scaleFactor in range(101, 201, 1):
        for minNeighbors in range(1, 11, 1):
            results = detect_eyes(img_arr[i], scaleFactor/100, minNeighbors)

            #print('length: ', len(results))

            lx,ly,rx,ry = None,None,None,None

            if len(results) == 4:
                lx,ly,rx,ry = results
            elif len(results) == 2:
                lx,ly = results
                
            glx, gly = left[i]
            grx, gry = right[i]

            
            dl, dr = measureDistance(lx, ly, glx,gly, rx, ry, grx, gry)
            
            avg = (dl+dr)/2

            r[i][scaleFactor-101][minNeighbors-1] = avg

In [ ]:
#########################################
#  Use only for Problem 6
#########################################

# 3d array to store results (avg distances)
r = [[[0 for i in range(1)] for j in range(1)] for k in range(5)]


for i in range(0,5):
    # uncomment the following for problem 6:
    results = detect_eyes(img_arr[i])
    #results = detect_eyes(img_arr[i], scaleFactor/100, minNeighbors)

    #print('length: ', len(results))

    lx,ly,rx,ry = None,None,None,None

    if len(results) == 4:
        lx,ly,rx,ry = results
    elif len(results) == 2:
        lx,ly = results
        
    glx, gly = left[i]
    grx, gry = right[i]

    
    dl, dr = measureDistance(lx, ly, glx,gly, rx, ry, grx, gry)
    
    avg = (dl+dr)/2

    r[i][0][0] = avg

df = pd.DataFrame(index=[10],columns=["1.84"])
avg = [[0 for j in range(1)] for i in range(1)]



for i in range(0,5):
    avg[0][0] += r[i][0][0]

#print(f"minNeighbors: {minNeighbors-1}, scaleFactor: {scaleFactor-101}, avg: {avg[minNeighbors-1][scaleFactor-101]/5}")
df.iat[0,0] = (avg[0][0] / 5)

df.to_excel('eye_detection_results-optimized.xlsx', header=True)

In [33]:

df = pd.DataFrame(index=[k for k in range(1,11,1)],columns=[j/100 for j in range(101,201,1)])
avg = [[0 for j in range(100)] for i in range(10)]



for minNeighbors in range(1, 11, 1):
    for scaleFactor in range(101, 201, 1):
        for i in range(0,5):
            avg[minNeighbors-1][scaleFactor-101] += r[i][scaleFactor-101][minNeighbors-1]

        print(f"minNeighbors: {minNeighbors-1}, scaleFactor: {scaleFactor-101}, avg: {avg[minNeighbors-1][scaleFactor-101]/5}")
        df.iat[minNeighbors-1, scaleFactor-101] = (avg[minNeighbors-1][scaleFactor-101] / 5)

df.to_excel('eye_detection_results.xlsx', header=True)

minNeighbors: 0, scaleFactor: 0, avg: 440.3939329852715
minNeighbors: 0, scaleFactor: 1, avg: 501.51374798551444
minNeighbors: 0, scaleFactor: 2, avg: 462.2932871336475
minNeighbors: 0, scaleFactor: 3, avg: 503.6644353313861
minNeighbors: 0, scaleFactor: 4, avg: 509.8228039020467
minNeighbors: 0, scaleFactor: 5, avg: 511.9224736783108
minNeighbors: 0, scaleFactor: 6, avg: 492.53891780484844
minNeighbors: 0, scaleFactor: 7, avg: 505.3427774656331
minNeighbors: 0, scaleFactor: 8, avg: 500.26025881896066
minNeighbors: 0, scaleFactor: 9, avg: 502.5765111329859
minNeighbors: 0, scaleFactor: 10, avg: 490.8120615766293
minNeighbors: 0, scaleFactor: 11, avg: 495.080287761631
minNeighbors: 0, scaleFactor: 12, avg: 503.46341482545375
minNeighbors: 0, scaleFactor: 13, avg: 496.93478241320355
minNeighbors: 0, scaleFactor: 14, avg: 494.42979230023974
minNeighbors: 0, scaleFactor: 15, avg: 493.91719709188294
minNeighbors: 0, scaleFactor: 16, avg: 515.7511149992484
minNeighbors: 0, scaleFactor: 17, a

In [34]:
# Problem 5
# Find the position of the minimum value
min_value = df.min().min()  # Minimum value in the entire DataFrame
min_pos = df.stack().idxmin()  # Returns (row_index, column_name) of min value

print(f"Minimum value: {min_value}")
print(f"Row index (minNeighbors): {min_pos[0]}")
print(f"Column name (scaleFactor): {min_pos[1]}")

Minimum value: 82.19947062650111
Row index (minNeighbors): 10
Column name (scaleFactor): 1.84
